Rizzbot notebook 1: Data Preprocessing pipeline. The purpose of this notebook is to set up the pinecone index, scrape the YT videos from the channel of choice, convert 
the YT videos into Vectors and then store the vectors into the pinecone index for later use. For each step in between, we save the files temporarily on my local device and then transfer them to a bucket storage in AWS. All vectors are stored as 384 dimensions.  

In [ ]:
# Set up and connect to the environment for Pinecone
import os
from dotenv import load_dotenv, find_dotenv
from pinecone import Pinecone, ServerlessSpec

# Load environment variables from .env file
_ = load_dotenv(find_dotenv())
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")

# Initialize Pinecone client
pc = Pinecone(api_key = PINECONE_API_KEY)

if index_name not in pc.list_indexes().names():
    pc.create_index(
        name=index_name,
        dimension=dimension,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )

# Connect to the index
index = pc.Index(index_name)

print(f"Connected to index: {index_name}")

Connected to index: rizzbot


In [1]:
# Script with batch processing and timeout, downloaded 200 videos after which I manually interrupted the script. Because 200 videos is enough and youtube has started blocking the script
import subprocess
import boto3
import time
import json
import sys
from pathlib import Path


# === CONFIG ===
CHANNEL_URL = "https://www.youtube.com/user/Charismaoncommand/videos"
# OR use single video
VIDEO_URL = "https://www.youtube.com/watch?v=oIiv_335yus"

# Choose mode: 'channel' or 'video'
MODE = "video"  # Change to "video" to process single video

# Audio chunking settings
CHUNK_DURATION = 1800  # 30 minutes per chunk (in seconds)
ENABLE_CHUNKING = True  # Set to False to disable chunking

TEMP_DIR = (Path.cwd() / "tmp_files" / "rizzbot_downloads").resolve()
S3_BUCKET = "rizzbot-temp-storage"
S3_PREFIX = "rizzbot/Audio"
YT_TIMEOUT = 600  # 10 minutes max per video
SLEEP_MIN = 2
SLEEP_MAX = 5
MAX_VIDEOS = 397  # actual video count

session = boto3.Session(profile_name="rizzbot")
s3_client = session.client("s3")
TEMP_DIR.mkdir(parents=True, exist_ok=True)

def log(msg):
    print(f"[RIZZBOT] {msg}")

def get_video_list(channel_url, max_videos):
    """
    Use yt-dlp to fetch video URLs from the channel
    """
    try:
        result = subprocess.run(
            ["yt-dlp", "--flat-playlist", "--dump-single-json", "--playlist-end", str(max_videos), channel_url],
            capture_output=True, text=True, check=True
        )
        playlist = json.loads(result.stdout)  # yt-dlp returns JSON, safer to parse but yt-dlp can be inconsistent
        entries = playlist.get("entries", [])
        video_urls = [f"https://www.youtube.com/watch?v={entry['id']}" for entry in entries]
        return video_urls
    except subprocess.CalledProcessError as e:
        log(f"Failed to fetch video list: {e.stderr}")
        sys.exit(1)

def get_audio_duration(audio_file):
    """Get duration of audio file in seconds using ffprobe"""
    try:
        result = subprocess.run([
            r"C:\Users\karel\ffmpeg-7.1.1-full_build\bin\ffprobe",
            "-v", "quiet",
            "-show_entries", "format=duration",
            "-of", "csv=p=0",
            str(audio_file)
        ], capture_output=True, text=True, check=True)
        return float(result.stdout.strip())
    except (subprocess.CalledProcessError, ValueError) as e:
        log(f"Failed to get duration for {audio_file}: {e}")
        return 0

def chunk_audio_file(audio_file, chunk_duration):
    """Split audio file into chunks using ffmpeg"""
    base_name = audio_file.stem
    chunks = []
    
    duration = get_audio_duration(audio_file)
    if duration <= chunk_duration:
        log(f"Audio file {audio_file.name} is {duration:.1f}s, no chunking needed")
        return [audio_file]
    
    log(f"Chunking {audio_file.name} ({duration:.1f}s) into {chunk_duration}s segments...")
    
    chunk_num = 0
    start_time = 0
    
    while start_time < duration:
        chunk_num += 1
        chunk_file = audio_file.parent / f"{base_name}_part{chunk_num:02d}.mp3"
        
        ffmpeg_cmd = [
            r"C:\Users\karel\ffmpeg-7.1.1-full_build\bin\ffmpeg",
            "-i", str(audio_file),
            "-ss", str(start_time),
            "-t", str(chunk_duration),
            "-c", "copy",  # Copy without re-encoding for speed
            "-avoid_negative_ts", "make_zero",
            str(chunk_file),
            "-y"  # Overwrite output files
        ]
        
        try:
            subprocess.run(ffmpeg_cmd, check=True, capture_output=True, text=True)
            chunks.append(chunk_file)
            log(f"Created chunk {chunk_num}: {chunk_file.name}")
        except subprocess.CalledProcessError as e:
            log(f"Failed to create chunk {chunk_num}: {e}")
            break
        
        start_time += chunk_duration
    
    # Remove original file after successful chunking
    if chunks:
        audio_file.unlink()
        log(f"Removed original file {audio_file.name}, created {len(chunks)} chunks")
    
    return chunks

def download_and_upload_video(video_url, index, total):
    """
    Download single video audio and upload to S3
    """
    log(f"Processing video {index + 1}/{total}: {video_url}")

    output_path = str(TEMP_DIR / "%(title)s.%(ext)s")

    ytdlp_cmd = [
        "yt-dlp",
        "--extract-audio",
        "--audio-format", "mp3",
        "--ignore-errors",
        "--socket-timeout", "10",
        "--sleep-interval", str(SLEEP_MIN),
        "--max-sleep-interval", str(SLEEP_MAX),
        "--ffmpeg-location", r"C:\Users\karel\ffmpeg-7.1.1-full_build\bin",
        "-o", output_path,
        video_url
    ]

    try:
        subprocess.run(ytdlp_cmd, check=True, timeout=YT_TIMEOUT, capture_output=True, text=True)
    except subprocess.TimeoutExpired:
        log(f"Timeout expired for video: {video_url}. Skipping.")
        return
    except subprocess.CalledProcessError as e:
        log(f"yt-dlp failed for video {video_url}: {e.stderr}")
        return

    # Process downloaded audio files (original or chunks)
    audio_files = list(TEMP_DIR.glob("*.mp3"))
    
    if not audio_files:
        log(f"No audio files downloaded for {video_url}")
        return
    
    # Chunk long audio files if enabled
    if ENABLE_CHUNKING:
        all_chunks = []
        for audio_file in audio_files:
            chunks = chunk_audio_file(audio_file, CHUNK_DURATION)
            all_chunks.extend(chunks)
        audio_files = all_chunks
    
    # Upload all audio files (original or chunks)
    for audio_file in audio_files:
        s3_key = f"{S3_PREFIX}/{audio_file.name}"
        try:
            s3_client.upload_file(str(audio_file), S3_BUCKET, s3_key)
            log(f"Uploaded {audio_file.name} to s3://{S3_BUCKET}/{s3_key}")
        except Exception as e:
            log(f"Failed to upload {audio_file.name}: {e}")
        audio_file.unlink()

def process_videos():
    """Process videos based on selected mode (channel or single video)"""
    if MODE == "channel":
        log(f"Processing channel: {CHANNEL_URL}")
        video_urls = get_video_list(CHANNEL_URL, MAX_VIDEOS)
        total = len(video_urls)
        log(f"Found {total} videos to process from channel.")
        
        for index, video_url in enumerate(video_urls):
            download_and_upload_video(video_url, index, total)
            
    elif MODE == "video":
        log(f"Processing single video: {VIDEO_URL}")
        download_and_upload_video(VIDEO_URL, 0, 1)
        
    else:
        log(f"Invalid MODE: {MODE}. Please set MODE to 'channel' or 'video'")
        return
    
    log("All videos processed.")

# === RUN ===
if __name__ == "__main__":
    process_videos()


[RIZZBOT] Processing single video: https://www.youtube.com/watch?v=oIiv_335yus
[RIZZBOT] Processing video 1/1: https://www.youtube.com/watch?v=oIiv_335yus
[RIZZBOT] Chunking No. 1 Communication Expert： This Speaking Mistake Makes People Dislike You! Vinh Giang.mp3 (8808.3s) into 1800s segments...
[RIZZBOT] Created chunk 1: No. 1 Communication Expert： This Speaking Mistake Makes People Dislike You! Vinh Giang_part01.mp3
[RIZZBOT] Created chunk 2: No. 1 Communication Expert： This Speaking Mistake Makes People Dislike You! Vinh Giang_part02.mp3
[RIZZBOT] Created chunk 3: No. 1 Communication Expert： This Speaking Mistake Makes People Dislike You! Vinh Giang_part03.mp3
[RIZZBOT] Created chunk 4: No. 1 Communication Expert： This Speaking Mistake Makes People Dislike You! Vinh Giang_part04.mp3
[RIZZBOT] Created chunk 5: No. 1 Communication Expert： This Speaking Mistake Makes People Dislike You! Vinh Giang_part05.mp3
[RIZZBOT] Removed original file No. 1 Communication Expert： This Speaking Mis

In [1]:
# This script downloads audio files from S3, transcribes them using OpenAI's Whisper model, and uploads the transcripts back to a different folder in S3.


import os
import openai
import boto3
import json
import time
import tempfile
from pathlib import Path
from dotenv import load_dotenv, find_dotenv
from concurrent.futures import ThreadPoolExecutor, as_completed
from botocore.exceptions import BotoCoreError, ClientError
import traceback

_ = load_dotenv(find_dotenv())

# === CONFIG ===
S3_BUCKET = "rizzbot-temp-storage"
AUDIO_PREFIX = "rizzbot/Audio/"
TRANSCRIPT_PREFIX = "rizzbot/Transcripts"

# Choose transcription mode: 'all', 'latest', or 'today'
TRANSCRIPTION_MODE = "today"  # Change to "all" to process all audio files, "latest" for single newest file, "today" for all files uploaded today

api_key = os.getenv("OPENAI_TEST_KEY_KdR")
openai.api_key = api_key
MAX_RETRIES = 3
MAX_WORKERS = 3  # Number of concurrent transcriptions

session = boto3.Session(profile_name="rizzbot")
s3_client = session.client("s3")


def log(msg):
    print(f"[RIZZBOT] {msg}")


def download_file_with_retry(bucket, key, local_path):
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            s3_client.download_file(bucket, key, str(local_path))
            return
        except (BotoCoreError, ClientError) as e:
            log(f"Download failed (attempt {attempt}): {e}")
            time.sleep(2 * attempt)
    raise RuntimeError(f"Failed to download {key} after {MAX_RETRIES} attempts.")


def upload_file_with_retry(bucket, key, body_bytes):
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            s3_client.put_object(Bucket=bucket, Key=key, Body=body_bytes)
            return
        except (BotoCoreError, ClientError) as e:
            log(f"Upload failed (attempt {attempt}): {e}")
            time.sleep(2 * attempt)
    raise RuntimeError(f"Failed to upload {key} after {MAX_RETRIES} attempts.")


def transcribe_audio_from_s3(s3_bucket, s3_key):
    # Use a unique tmp file path
    with tempfile.NamedTemporaryFile(suffix=".mp3", delete=False) as tmp_file:
        local_file = Path(tmp_file.name)

    download_file_with_retry(s3_bucket, s3_key, local_file)

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            with open(local_file, "rb") as audio_file:
                transcript_resp = openai.audio.transcriptions.create(
                    model="whisper-1",
                    file=audio_file,
                    response_format="json"
                )
            break
        except openai.OpenAIError as e:
            log(f"Transcription failed (attempt {attempt}): {e}")
            time.sleep(2 * attempt)
    else:
        local_file.unlink(missing_ok=True)
        raise RuntimeError(f"Failed to transcribe {s3_key} after {MAX_RETRIES} attempts.")

    local_file.unlink(missing_ok=True)
    return transcript_resp


def save_transcript_to_s3(transcript_data, s3_bucket, s3_key):
    json_bytes = json.dumps(transcript_data.model_dump()).encode('utf-8')
    upload_file_with_retry(s3_bucket, s3_key, json_bytes)
    log(f"Transcript raw: {transcript_data.model_dump()}")


def process_single_audio(s3_key):
    log(f"Starting transcription for {s3_key}")
    try:
        transcript_data = transcribe_audio_from_s3(S3_BUCKET, s3_key)
        base_name = Path(s3_key).stem
        transcript_s3_key = f"{TRANSCRIPT_PREFIX}/{base_name}.json"
        save_transcript_to_s3(transcript_data, S3_BUCKET, transcript_s3_key)
        log(f"Completed transcription for {s3_key}")
    except Exception as e:
        log(f"Transcription failed for {s3_key}: {e}")
        log(traceback.format_exc())


def get_audio_files_to_process():
    """Get audio files based on selected mode"""
    log(f"Listing audio files in s3://{S3_BUCKET}/{AUDIO_PREFIX}")

    paginator = s3_client.get_paginator('list_objects_v2')
    pages = paginator.paginate(Bucket=S3_BUCKET, Prefix=AUDIO_PREFIX)

    mp3_files = []
    for page in pages:
        for obj in page.get("Contents", []):
            if obj["Key"].endswith(".mp3"):
                mp3_files.append({
                    'key': obj["Key"],
                    'last_modified': obj['LastModified']
                })

    if not mp3_files:
        log("No MP3 files found to process.")
        return []

    if TRANSCRIPTION_MODE == "latest":
        # Sort by last modified time and get the most recent file
        mp3_files.sort(key=lambda x: x['last_modified'], reverse=True)
        latest_file = mp3_files[0]
        log(f"Latest file found: {latest_file['key']} (modified: {latest_file['last_modified']})")
        return [latest_file['key']]
    
    elif TRANSCRIPTION_MODE == "today":
        # Get all files uploaded today
        from datetime import datetime, timezone
        today = datetime.now(timezone.utc).date()
        
        today_files = []
        for file in mp3_files:
            file_date = file['last_modified'].date()
            if file_date == today:
                today_files.append(file)
        
        if today_files:
            # Sort by modification time for consistent processing order
            today_files.sort(key=lambda x: x['last_modified'])
            file_keys = [file['key'] for file in today_files]
            log(f"Found {len(file_keys)} MP3 files uploaded today ({today}):")
            for file in today_files:
                log(f"  - {file['key']} (modified: {file['last_modified']})")
            return file_keys
        else:
            log(f"No MP3 files found uploaded today ({today})")
            return []
    
    elif TRANSCRIPTION_MODE == "all":
        file_keys = [file['key'] for file in mp3_files]
        log(f"Found {len(file_keys)} MP3 files to process.")
        return file_keys
    
    else:
        log(f"Invalid TRANSCRIPTION_MODE: {TRANSCRIPTION_MODE}. Please set to 'all', 'latest', or 'today'")
        return []


def process_audio_files():
    """Process audio files based on selected mode"""
    mp3_files = get_audio_files_to_process()
    
    if not mp3_files:
        return

    log(f"Starting transcription of {len(mp3_files)} file(s)...")

    if TRANSCRIPTION_MODE == "latest" and len(mp3_files) == 1:
        # Process single file directly
        process_single_audio(mp3_files[0])
    else:
        # Process multiple files with threading
        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
            futures = {executor.submit(process_single_audio, key): key for key in mp3_files}
            for future in as_completed(futures):
                key = futures[future]
                try:
                    future.result()
                except Exception as e:
                    log(f"Unhandled failure for {key}: {e}")
                    log(traceback.format_exc())


process_audio_files()

[RIZZBOT] Listing audio files in s3://rizzbot-temp-storage/rizzbot/Audio/
[RIZZBOT] Found 5 MP3 files uploaded today (2025-10-31):
[RIZZBOT]   - rizzbot/Audio/No. 1 Communication Expert： This Speaking Mistake Makes People Dislike You! Vinh Giang_part01.mp3 (modified: 2025-10-31 15:06:26+00:00)
[RIZZBOT]   - rizzbot/Audio/No. 1 Communication Expert： This Speaking Mistake Makes People Dislike You! Vinh Giang_part02.mp3 (modified: 2025-10-31 15:06:30+00:00)
[RIZZBOT]   - rizzbot/Audio/No. 1 Communication Expert： This Speaking Mistake Makes People Dislike You! Vinh Giang_part03.mp3 (modified: 2025-10-31 15:06:33+00:00)
[RIZZBOT]   - rizzbot/Audio/No. 1 Communication Expert： This Speaking Mistake Makes People Dislike You! Vinh Giang_part04.mp3 (modified: 2025-10-31 15:06:36+00:00)
[RIZZBOT]   - rizzbot/Audio/No. 1 Communication Expert： This Speaking Mistake Makes People Dislike You! Vinh Giang_part05.mp3 (modified: 2025-10-31 15:06:39+00:00)
[RIZZBOT] Starting transcription of 5 file(s)...


In [ ]:
# This script chunks and embeds transcripts from S3, uploading them to Pinecone.
# It uses OpenAI's embedding model and handles retries with exponential backoff.
# Vector IDs are sequential (emb-0001, emb-0002, etc.) for easy retrieval.
# Vector dimensions can be set, matching the required model output, for this version we stick with 384 dimensions. 

import json
import time
import argparse
import yaml
from concurrent.futures import ThreadPoolExecutor, as_completed
from threading import Lock

import boto3
import openai
from pinecone import Pinecone
import tiktoken
from tqdm import tqdm
from dotenv import load_dotenv, find_dotenv
from pathlib import Path
from sentence_transformers import SentenceTransformer

# --- INIT ---
session = boto3.Session(profile_name="rizzbot", region_name="eu-north-1")
s3_client = session.client("s3")

# Global counter for vector IDs with thread safety
vector_counter = 0
counter_lock = Lock()

def get_next_vector_id():
    """Generate sequential vector IDs like emb-0001, emb-0002, etc."""
    global vector_counter
    with counter_lock:
        vector_counter += 1
        return f"emb-{vector_counter:04d}"

# --- RETRY DECORATOR ---
def retry_with_backoff(max_retries=5, initial_delay=1.0, backoff_factor=2.0):
    def decorator(func):
        def wrapper(*args, **kwargs):
            delay = initial_delay
            for attempt in range(max_retries):
                try:
                    return func(*args, **kwargs)
                except Exception as e:
                    if attempt == max_retries - 1:
                        raise
                    print(f"Retry {attempt+1}/{max_retries} after error: {e}")
                    time.sleep(delay)
                    delay *= backoff_factor
        return wrapper
    return decorator

# --- CHUNKING ---
def chunk_text_tokens(text, chunk_size, overlap, tokenizer):
    tokens = tokenizer.encode(text)
    chunks = []
    start = 0
    while start < len(tokens):
        end = start + chunk_size
        chunk_tokens = tokens[start:end]
        chunk_text = tokenizer.decode(chunk_tokens)
        chunks.append(chunk_text)
        start += chunk_size - overlap
    return chunks

# --- LOAD TRANSCRIPTS ---
def list_transcript_keys(s3_client, bucket, prefix):
    keys = []
    continuation_token = None
    while True:
        kwargs = {'Bucket': bucket, 'Prefix': prefix}
        if continuation_token:
            kwargs['ContinuationToken'] = continuation_token
        response = s3_client.list_objects_v2(**kwargs)
        keys.extend([item['Key'] for item in response.get('Contents', []) if item['Key'].endswith('.json')])
        if response.get('IsTruncated'):
            continuation_token = response.get('NextContinuationToken')
        else:
            break
    return keys

def load_transcript(s3_client, bucket, key):
    response = s3_client.get_object(Bucket=bucket, Key=key)
    body = response['Body'].read()
    data = json.loads(body)
    return data.get('text', '')

# --- EMBEDDING + UPLOAD ---
@retry_with_backoff()
def embed_chunk(chunk, embed_model, embed_dimensions):
    """
    Generate embeddings for a text chunk.
    Now uses sentence-transformers/all-MiniLM-L6-v2 model which outputs 384-dimensional embeddings.
    Note: This model produces 384-dim embeddings, not 1536-dim like OpenAI models.
    """
    
    # Initialize the sentence transformer model
    model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
    
    # Generate embedding
    embedding = model.encode(chunk, convert_to_tensor=False).tolist()
    
    # Note: all-MiniLM-L6-v2 produces 384-dimensional embeddings
    actual_dimensions = len(embedding)
    if actual_dimensions != embed_dimensions:
        print(f"Warning: Expected {embed_dimensions} dimensions, got {actual_dimensions}")
        print(f"Note: all-MiniLM-L6-v2 model produces 384-dimensional embeddings")
    
    return embedding

@retry_with_backoff()
def upsert_batch(index, batch):
    index.upsert(batch)

# --- PROCESS TRANSCRIPT ---
def process_transcript(s3_client, key, config, tokenizer, index, model=None):
    try:
        # Load transcript text
        text = load_transcript(s3_client, config['s3_bucket'], key)
        
        if not text.strip():
            print(f"Empty transcript for {key}, skipping")
            return 0
        
        # Chunk the text
        if model is not None:
            # For Sentence Transformers, use simple character-based chunking
            # since tokenizer might not have encode/decode methods
            chunk_size = config.get('chunk_size', 512)  # characters
            overlap = config.get('overlap', 50)  # characters
            
            text_chunks = []
            start = 0
            while start < len(text):
                end = start + chunk_size
                chunk = text[start:end]
                text_chunks.append(chunk)
                start += chunk_size - overlap
                if end >= len(text):
                    break
        else:
            # For OpenAI models, use token-based chunking
            chunk_size = config.get('chunk_size', 512)  # tokens
            overlap = config.get('overlap', 50)  # tokens
            text_chunks = chunk_text_tokens(text, chunk_size, overlap, tokenizer)
        
        if not text_chunks:
            print(f"No chunks created for {key}, skipping")
            return 0
        
        # Generate embeddings and prepare for upsert
        vectors_to_upsert = []
        
        for i, chunk in enumerate(text_chunks):
            if not chunk.strip():
                continue
                
            # Generate embedding
            if model is not None:
                # Use Sentence Transformer
                embedding = model.encode(chunk, convert_to_numpy=True).tolist()
            else:
                # Use OpenAI API (implement if needed)
                embedding = embed_chunk(chunk, config['embed_model'], config['embed_dimensions'])
            
            # Create vector data
            vector_id = get_next_vector_id()
            metadata = {
                's3_key': key,
                'chunk_index': i,
                'text': chunk[:500],  # Truncate for metadata size limits
                'full_text_length': len(chunk)
            }
            
            vectors_to_upsert.append({
                'id': vector_id,
                'values': embedding,
                'metadata': metadata
            })
        
        # Upsert in batches
        batch_size = config.get('batch_size', 100)
        for i in range(0, len(vectors_to_upsert), batch_size):
            batch = vectors_to_upsert[i:i + batch_size]
            upsert_batch(index, batch)
        
        print(f"Processed {key}: {len(vectors_to_upsert)} vectors created")
        return len(vectors_to_upsert)
        
    except Exception as e:
        print(f"Error processing {key}: {e}")
        import traceback
        traceback.print_exc()
        return 0

# --- HELPER FUNCTIONS FOR RETRIEVAL ---
def get_vector_id_range(start_id, end_id):
    """Generate a list of vector IDs in range for easy retrieval"""
    start_num = int(start_id.split('-')[1])
    end_num = int(end_id.split('-')[1])
    return [f"emb-{i:04d}" for i in range(start_num, end_num + 1)]

def save_vector_mapping(vectors_created, config_path):
    """Save a mapping file to track which vectors belong to which transcripts"""
    mapping_file = Path(config_path).parent / 'vector_mapping.json'
    mapping = {
        'total_vectors': vector_counter,
        'last_vector_id': f"emb-{vector_counter:04d}",
        'vectors_per_transcript': vectors_created,
        'embedding_dimensions': 1536  # Document the dimensions used
    }
    with open(mapping_file, 'w') as f:
        json.dump(mapping, f, indent=2)
    print(f"Vector mapping saved to {mapping_file}")

index_name = "rizzbot-384"  # Updated index name for 384 dimensions
dimensions = 384


# --- MAIN ---
def main(config_path, s3_client):
    global vector_counter
    
    # Load config
    with open(config_path, 'r') as f:
        config = yaml.safe_load(f)

    # Validate embedding dimensions
    if config.get('embed_dimensions', 384) != 384:
        print(f"Warning: Config specifies {config.get('embed_dimensions')} dimensions, forcing to 384")
        config['embed_dimensions'] = 384

    # Updated model validation for Sentence Transformers
    valid_models = [
        'all-MiniLM-L6-v2',        # 384 dimensions
        'all-mpnet-base-v2',       # 768 dimensions  
        'llama-text-embed-v2'      # Configurable
    ]
    
    if config['embed_model'] not in valid_models:
        print(f"Warning: Model {config['embed_model']} may not support 384 dimensions")
        print(f"Recommended models for 384 dimensions: {valid_models}")

    # Remove OpenAI API key if using Sentence Transformers
    if config['embed_model'] in ['all-MiniLM-L6-v2', 'all-mpnet-base-v2']:
        # Load Sentence Transformer model
        from sentence_transformers import SentenceTransformer
        model = SentenceTransformer(config['embed_model'])
        tokenizer = model.tokenizer  # Use the model's tokenizer
    else:
        # For OpenAI models
        openai.api_key = config['openai_api_key']
        tokenizer = tiktoken.encoding_for_model(config['embed_model'])
        model = None

    pc = Pinecone(api_key=config['pinecone_api_key'])
    index = pc.Index(config['pinecone_index'])

    keys = list_transcript_keys(s3_client, config['s3_bucket'], config['transcripts_prefix'])
    
    # Track vectors created per transcript
    vectors_created = {}
    total_vectors = 0

    print(f"Using embedding model: {config['embed_model']}")
    print(f"Target embedding dimensions: {config['embed_dimensions']}")
    print(f"Processing {len(keys)} transcripts...")

    with ThreadPoolExecutor(max_workers=config['max_workers']) as executor:
        futures = {executor.submit(process_transcript, s3_client, key, config, tokenizer, index, model): key for key in keys}
        for future in tqdm(as_completed(futures), total=len(futures), desc='Processing transcripts'):
            try:
                key = futures[future]
                num_vectors = future.result()
                vectors_created[key] = num_vectors
                total_vectors += num_vectors
                print(f"Processed {key}: {num_vectors} vectors (384 dims each)")
            except Exception as e:
                print(f"Failed processing {futures[future]}: {e}")
    
    print(f"\nTotal vectors created: {total_vectors}")
    print(f"Vector ID range: emb-0001 to emb-{vector_counter:04d}")
    print(f"All vectors saved with 384 dimensions")
    
    # Save mapping for future reference
    save_vector_mapping(vectors_created, config_path)
    
# --- RETRIEVAL HELPER FUNCTIONS ---
def retrieve_all_vectors(config_path):
    """Retrieve all vectors for clustering - call this in your next notebook"""
    with open(config_path, 'r') as f:
        config = yaml.safe_load(f)
    
    pc = Pinecone(api_key=config['pinecone_api_key'])
    index = pc.Index(config['pinecone_index'])
    
    # Load vector mapping to know the range
    mapping_file = Path(config_path).parent / 'vector_mapping.json'
    with open(mapping_file, 'r') as f:
        mapping = json.load(f)
    
    total_vectors = mapping['total_vectors']
    vector_ids = [f"emb-{i:04d}" for i in range(1, total_vectors + 1)]
    
    # Fetch in batches (Pinecone has limits on batch size)
    batch_size = 100
    all_vectors = []
    
    for i in range(0, len(vector_ids), batch_size):
        batch_ids = vector_ids[i:i + batch_size]
        response = index.fetch(ids=batch_ids)
        all_vectors.extend(list(response['vectors'].values()))
    
    return all_vectors

def retrieve_vectors_by_transcript(config_path, s3_key):
    """Retrieve all vectors for a specific transcript"""
    with open(config_path, 'r') as f:
        config = yaml.safe_load(f)
    
    pc = Pinecone(api_key=config['pinecone_api_key'])
    index = pc.Index(config['pinecone_index'])
    
    # Query by metadata
    response = index.query(
        filter={"s3_key": s3_key},
        top_k=10000,  # Adjust based on expected chunk count
        include_metadata=True,
        include_values=True
    )
    
    return response['matches']

# --- CLI ---
if __name__ == '__main__':
    # Check if running in Jupyter notebook
    try:
        get_ipython()
        # Running in Jupyter - use default config path
        config_path = str(Path.cwd() / 'config.yaml')
        print(f"Running in Jupyter notebook, using config: {config_path}")
        main(config_path, s3_client)
    except NameError:
        # Running from command line - use argparse
        parser = argparse.ArgumentParser(description="Rizzbot Chunk + Embed Pipeline")
        parser.add_argument('--config', type=str, required=True, help='Path to YAML config file')
        args = parser.parse_args()
        main(args.config, s3_client)


Running in Jupyter notebook, using config: c:\Users\karel\Ironhack-Bootcamp-Assignments\Rizzbot\config.yaml
Using embedding model: all-MiniLM-L6-v2
Target embedding dimensions: 384
Processing 179 transcripts...


Processing transcripts:   1%|          | 2/179 [00:03<04:47,  1.63s/it]

Processed rizzbot/Transcripts/3 Jokes To Make Rude People Regret Insulting You.json: 20 vectors created
Processed rizzbot/Transcripts/3 Jokes To Make Rude People Regret Insulting You.json: 20 vectors (384 dims each)
Processed rizzbot/Transcripts/3 Common Jokes That Make People INSTANTLY Like You Less.json: 27 vectors created
Processed rizzbot/Transcripts/3 Common Jokes That Make People INSTANTLY Like You Less.json: 27 vectors (384 dims each)


Processing transcripts:   2%|▏         | 3/179 [00:04<02:47,  1.05it/s]

Processed rizzbot/Transcripts/$120,000 Was Stolen From Me... It Was My ＂Friend＂.json: 46 vectors created
Processed rizzbot/Transcripts/$120,000 Was Stolen From Me... It Was My ＂Friend＂.json: 46 vectors (384 dims each)


...

Processed rizzbot/Transcripts/Words That Win： How To Instantly Influence Anyone (use ethically).json: 34 vectors created
Processed rizzbot/Transcripts/Words That Win： How To Instantly Influence Anyone (use ethically).json: 34 vectors (384 dims each)
Processed rizzbot/Transcripts/Women are easily attracted to men who have these habits.json: 41 vectors created
Processed rizzbot/Transcripts/Women are easily attracted to men who have these habits.json: 41 vectors (384 dims each)


Processing transcripts:  99%|█████████▉| 178/179 [00:43<00:00,  4.76it/s]

Processed rizzbot/Transcripts/You Think You’re Charismatic… But This Proves You’re Just a People Pleaser.json: 23 vectors created
Processed rizzbot/Transcripts/You Think You’re Charismatic… But This Proves You’re Just a People Pleaser.json: 23 vectors (384 dims each)


Processing transcripts: 100%|██████████| 179/179 [00:44<00:00,  4.05it/s]

Processed rizzbot/Transcripts/why you feel awkward in conversation.json: 32 vectors created
Processed rizzbot/Transcripts/why you feel awkward in conversation.json: 32 vectors (384 dims each)

Total vectors created: 5099
Vector ID range: emb-0001 to emb-5099
All vectors saved with 384 dimensions
Vector mapping saved to c:\Users\karel\Ironhack-Bootcamp-Assignments\Rizzbot\vector_mapping.json
